# 一次 Agent 执行到底发生了什么？

## V0.1 Execution Kernel / Tool Boundary

这个 lab 的答案不是 `42`。真正的问题是：一个随手写的 tool-using loop，怎样变成 Kernel 可以观察、恢复、授权和审计的结构化执行？

**Prediction / question:** 如果模型只是提出 `math.add(20, 22)`，谁负责把“提议”变成可记录的 Tool boundary？

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the AgentKernel repository.")

REPOSITORY_ROOT = find_repo_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
LABS_ROOT = REPOSITORY_ROOT / "examples" / "labs"
if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))

from lab_helpers import event_rows, grant_rows, print_table, process_row, trajectory

## 1. Naive loop：能算出答案，但没有 runtime fact

In [ ]:
messages = ["user: What is 20 + 22?"]
proposal = {"tool": "math.add", "arguments": {"left": 20, "right": 22}}
tool_result = proposal["arguments"]["left"] + proposal["arguments"]["right"]
messages.append(f"tool result: {tool_result}")
messages.append(f"assistant: final answer: {tool_result}")
print_table([
    {"step": "model proposal", "observable": proposal},
    {"step": "tool result", "observable": tool_result},
    {"step": "stored facts", "observable": "only ad-hoc messages list"},
])

## 2. AgentKernel setup：同一个概念任务，穿过 ToolRegistry

In [ ]:
import asyncio
import json
from collections.abc import Mapping

from agentkernel import (
    Agent, DefaultAgentLoop, MessageRole, ModelRequest, ModelResponse,
    PromptService, ScriptedLLM, Session, ToolCall, ToolDefinition,
    ToolExecutionContext, ToolRegistry, ToolSchema,
)
from agentkernel.protocol import JsonValue

async def add(arguments: Mapping[str, JsonValue], _context: ToolExecutionContext) -> JsonValue:
    return int(arguments["left"]) + int(arguments["right"])

def final_answer(request: ModelRequest) -> ModelResponse:
    last = request.messages[-1]
    assert last.role is MessageRole.TOOL
    payload = json.loads(last.content)
    return ModelResponse(content=f"final answer: {payload['output']}")

session = Session("lab-v0-1-session")
agent = Agent.create(agent_id="lab-v0-1-agent", session=session, capabilities={"math.add"})
tools = ToolRegistry()
tools.register(ToolDefinition(
    schema=ToolSchema("math.add", "Add two integers.", {"type": "object"}),
    handler=add,
    required_capability="math.add",
))
llm = ScriptedLLM([
    ModelResponse(tool_calls=(ToolCall("call-add-1", "math.add", {"left": 20, "right": 22}),)),
    final_answer,
])
loop = DefaultAgentLoop(llm=llm, tools=tools, prompt=PromptService("Use the available tool."))
answer = asyncio.run(loop.run(agent, "What is 20 + 22?"))
answer

## 3. Inspect Kernel-visible execution facts

In [ ]:
print_table(event_rows(session))
trajectory(
    "User message",
    "Model proposal: tool/call math.add",
    "Kernel ToolRegistry boundary",
    "Tool result",
    "Session fact: tool/result",
    f"Assistant final answer: {answer}",
)

## Invariant

The model proposes; AgentKernel records the boundary crossing as structured Session facts.

## WHAT THIS DEMONSTRATES / 本实验验证什么

- A tool-using interaction becomes structured runtime execution.
- Session facts are the prerequisite for later recovery, authorization, accounting, and audit.

## WHAT THIS DOES NOT DEMONSTRATE / 本实验不证明什么

- It does not use a real model provider.
- It does not prove model reasoning quality.
- It does not prove crash recovery or side-effect safety.